In [7]:
# ------------------------------------------------------------
# Cutthroat: A Topology Optimization Framework
# Author: John A. Gardiner
# ------------------------------------------------------------

# This code is part of the Cutthroat topology optimization framework, which is designed to optimize material distribution in heat exchangers for improved performance. The framework includes mesh generation, finite element analysis, and optimization algorithms. The following code snippet focuses on the simulation of meshes and the integration of a neural network for predicting optimal material distributions.

In [8]:
# -------------------------------------------------------------
# Libraries and Imports
# -------------------------------------------------------------

# Paths and File Handling
import os
import subprocess
from pathlib import Path
from jinja2 import Template
import json
import uuid
from dataclasses import dataclass, field, asdict
from typing import Optional
import gmsh

# Numerical and Visualization Libraries
import numpy as np
import matplotlib.pyplot as plt
import pyvista as pv

# Neural Network and Machine Learning Libraries
import torch
import torch.nn as nn
from scipy.ndimage import label

In [9]:
class Generator:
    def __init__(self, domain_size, n_elements, generator_type, CNN_params=None):
        self.domain_length = domain_size[0]
        self.domain_height = domain_size[1]
        self.n_elements_x = n_elements[0]
        self.n_elements_y = n_elements[1]
        self.generator_type = generator_type
        self.mesh_min = min(self.domain_length / self.n_elements_x, self.domain_height / self.n_elements_y)
        self.mesh_max = max(self.domain_length / self.n_elements_x, self.domain_height / self.n_elements_y)
        self.mesh_params = {'mesh_algorithm': 8, 'mesh_recombine': 1, 'mesh_element_order': 2}
        #self.geo_generator = self.create_generator(generator_type, CNN_params)

        def create_generator(self, type, params):
            if type == 'CNN' and params is not None:
                # if CNN weights and biases are provided, we can initialize a CNN generator with those parameters
                return None
            elif type == 'CNN':
                # if CNN weights and biases are not provided, we can initialize a default CNN generator
                return None
            else:    
                # return a basic geometry generator
                return None
            
        def create_custom_hx(self, points):
            # create a custom hx obstruction in gmsh using the provided points
            n_points = len(points)
            gmsh_points = []
            for coord in points:
                point = gmsh.model.occ.addPoint(coord[0], coord[1], 0)
                gmsh_points.append(point)

            # Connect the points with lines to form the geometry of the obstruction
            lines = []
            for i in range(n_points):
                p1 = points[i]
                p2 = points[(i + 1) % n_points]  # Wrap around to connect the last point to the first
                line = gmsh.model.occ.addLine(p1, p2)
                lines.append(line)
            
            # create a surface from the lines
            cl = gmsh.model.occ.addCurveLoop(lines)
            surface = gmsh.model.occ.addPlaneSurface([cl])
            return surface
        
        def generate_mesh(self):
            # Initialize gmsh and create a new model
            gmsh.initialize()
            gmsh.model.add("heat_exchanger")

            # Use OpenCASCADE geometry kernel
            gmsh.model.occ

            # Define the domain geometry
            rect = gmsh.model.occ.addRectangle(0, -self.domain_height/2, 0, self.domain_length, self.domain_height)
            gmsh.model.occ.synchronize()

            # Add obstructions to the geometry based on the hx_objects provided by the generator
            obstructions = []
            for obj in self.hx_objects:
                obstruction = self.create_custom_hx(obj)
                obstructions.append(obstruction)
            gmsh.model.occ.synchronize()

            # Boolean cut
            cut = gmsh.model.occ.cut(
                [(2, rect)],
                [(2, s) for s in obstructions],
                removeObject=True,
                removeTool=True
            )
            gmsh.model.occ.synchronize()

            # Extract the resulting geometry after the cut operation
            outDimTags, _ = cut
            fluid_surfaces = [tag for dim, tag in outDimTags if dim == 2]

            if not fluid_surfaces:
                raise RuntimeError("Boolean cut resulted in no fluid domain. Please check the geometry of the obstructions.")
            
            gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")

            print("Cut result:", outDimTags)
            print("2D entities:", gmsh.model.getEntities(2))

            # Cylinder wall curves
            gmsh.model.occ.synchronize()
            boundaries = gmsh.model.getBoundary(
                [(2, s) for s in fluid_surfaces],
                oriented=False
            )

            obstruction_wall_curves = []
            inlet = []
            outlet = []
            top = []
            bottom = []

            for dim, tag in boundaries:
                com = gmsh.model.occ.getCenterOfMass(dim, tag)
                x, y, _ = com

                if abs(x) < 1e-6:
                    inlet.append(tag)
                elif abs(x - self.length) < 1e-6:
                    outlet.append(tag)
                elif abs(y - self.domain_height/2) < 1e-6:
                    top.append(tag)
                elif abs(y + self.domain_height/2) < 1e-6:
                    bottom.append(tag)
                else:
                    obstruction_wall_curves.append(tag)
            
            gmsh.model.addPhysicalGroup(1, obstruction_wall_curves, name="Wall")
            gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
            gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")
            gmsh.model.addPhysicalGroup(1, top, name="Top")
            gmsh.model.addPhysicalGroup(1, bottom, name="Bottom")
            
            # Set mesh parameters
            gmsh.option.setNumber("Mesh.CharacteristicLengthMin", self.mesh_params['mesh_min'])
            gmsh.option.setNumber("Mesh.CharacteristicLengthMax", self.mesh_params['mesh_max'])
            gmsh.option.setNumber("Mesh.Algorithm", self.mesh_params['mesh_algorithm']) # 8 
            gmsh.option.setNumber("Mesh.RecombineAll", self.mesh_params['mesh_recombine']) # 1
            gmsh.option.setNumber("Mesh.ElementOrder", self.mesh_params['mesh_element_order']) # 2

            # Generate the mesh
            gmsh.model.mesh.generate(2)
            gmsh.write(self.filename)
            gmsh.finalize()

In [10]:
class Simulator:
    def __init__(self, type='MOOSE', params="./cutthroat-opt"):
        self.type = type
        if self.type == 'MOOSE':
            self.app_name = params
        else:
            pass

    def run_moose(self, input_file, n_processors=1):
        '''Run a MOOSE simulation inside a subprocess
        
        Parameters:
        input_file: str, path to the MOOSE input file (.i)
        n_processors: int, number of processors to use for the simulation
        '''
        # Construct the command to run MOOSE
        cmd = [
            "conda", "run", "-n", "moose",
            "mpiexec", "-np", str(n_processors),
            self.app_name, "-i", input_file
        ]

        # Run the command and capture the output
        result = subprocess.run(cmd, capture_output=True, text=True)

        # Check for errors
        if result.returncode != 0:
            print("Error running MOOSE:")
            print(result.stderr)
        else:
            print("MOOSE simulation completed successfully.")
            print(result.stdout)

    def analyze_exodus_results(self, exodus_file):
        # Placeholder for analyzing Exodus results
        # This function would read the Exodus file, extract relevant data, and perform analysis
        mesh = pv.read(exodus_file)
        print(mesh)
        print(mesh.array_names)
    
    def run(self, input_file, n_processors=4):
        if self.type == 'MOOSE':
            self.run_moose(input_file, n_processors)
        else:
            pass

In [11]:
class Optimizer:
    def __init__(self, domain_dimensions, generator_type, simulator_type, mesh_files_dir, results_dir):
        self.generator = self.initialize_generator(generator_type, domain_dimensions)
        self.simulator = self.initialize_simulator(simulator_type)
        self.mesh_files_dir = mesh_files_dir
        self.results_dir = results_dir

    def initialize_generator(self, generator_type, domain_dimensions):
        gen = Generator(domain_size=domain_dimensions[0], n_elements=domain_dimensions[1], generator_type=generator_type)
        return gen
    
    def initialize_simulator(self, simulator_type):
        sim = Simulator(type=simulator_type)
        return sim

    def create_input_file(self, template_path, output_dir, mesh_name):
        '''Generate a MOOSE .i file from a template
        
        Parameters:
        - template_path: path to template.i file
        - output_dir: directory to save the generated .i file
        - mesh_name: name of the mesh file to read e.g. 'rd0001.msh'
        '''
        # Extract case prefix and number from mesh_name (assumes format 'rdXXXX.msh' or 'gdXXXX.msh')
        case_prefix = mesh_name.split('.')[0][:-4]
        case_number = mesh_name.split('.')[0][-4:]
        case_id = f"{case_prefix}{case_number}"

        input_filename = f"{case_id}.i"

        # Load the template
        with open(template_path, "r") as f:
            template = Template(f.read())
        
        # Render the template with the mesh name
        rendered = template.render(
            mesh_name=mesh_name,
            case_name=case_id,
            rho=1.0,  # Placeholder value for density
            mu=1.0,   # Placeholder value for viscosity
            k=1.0,    # Placeholder value for thermal conductivity
            cp=1.0    # Placeholder value for specific heat capacity
        )

        # Ensure output directory exists
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        # Write file to output directory
        output_path = output_dir / input_filename
        with open(output_path, "w") as f:
            f.write(rendered)
        
        return output_path

    def clean_input_file(self, input_file):
        '''Delete the generated .i file after simulation'''
        if os.path.exists(input_file):
            os.remove(input_file)

    def run_simulation(self, input_file, n_processors=None):
        '''Run the simulation using the optimizer's simulator
        
        Parameters:
        - input_file: path to the .i file to run
        - n_processors: number of processors to use for parallel execution if using MOOSE simulator (optional)
        '''
        if n_processors is not None:
            self.simulator.run(input_file, n_processors)
        else:
            self.simulator.run(input_file)

    def simulate_all_meshes(self):
        '''Simulate all meshes in the mesh_files_dir'''
        mesh_files = [f for f in os.listdir(self.mesh_files_dir) if f.endswith('.msh')]
        result_files = [r for r in os.listdir(self.results_dir) if r.endswith('.e')]
        for mesh_file in mesh_files:
            if any(mesh_file.split('.')[0] in result for result in result_files):
                print(f"Results already exist for mesh: {mesh_file}, skipping simulation.")
                continue
            print(f"Simulating for mesh: {mesh_file}")
            input_file = self.create_input_file(
                template_path="problems/template.i",
                output_dir=self.results_dir,
                mesh_name=mesh_file
            )
            self.run_simulation(input_file, n_processors=4)
            self.clean_input_file(input_file)
        

In [12]:
topology_optimizer = Optimizer(
    domain_dimensions=((1.0, 0.5), (50, 25)),
    generator_type='basic',
    simulator_type='MOOSE',
    mesh_files_dir="meshes",
    results_dir="results"
)
topology_optimizer.simulate_all_meshes()

Simulating for mesh: heat_exchanger.msh
Error running MOOSE:


*** ERROR ***
Unable to open file "/home/jagardiner/projects/cutthroat/results/../mesh/heat_exchanger.msh-mesh.cpa.gz". Check to make sure that it exists and that you have read permission.



*** ERROR ***
Unable to open file "/home/jagardiner/projects/cutthroat/results/../mesh/heat_exchanger.msh-mesh.cpa.gz". Check to make sure that it exists and that you have read permission.



*** ERROR ***
Unable to open file "/home/jagardiner/projects/cutthroat/results/../mesh/heat_exchanger.msh-mesh.cpa.gz". Check to make sure that it exists and that you have read permission.



*** ERROR ***
Unable to open file "/home/jagardiner/projects/cutthroat/results/../mesh/heat_exchanger.msh-mesh.cpa.gz". Check to make sure that it exists and that you have read permission.

Abort(1) on node 0 (rank 0 in comm 0): application called MPI_Abort(MPI_COMM_WORLD, 1) - process 0
terminate called without an active exception
Abort(1) on node 1 (rank 1 

In [13]:
@dataclass
class GeometryConfig:
    """
    Serializable snapshot of a heat exchanger geometry.
    
    Produced by the Generator and consumed by both the MeshGenerator (to build the .msh file)
    and the Optimizer (to map simulation results back onto the original grid).
    
    Fields:
    -------
    grid_nx, grid_ny: int
        Number of grid points in the x and y directions for the reference grid.
    domain_length, domain_height: float
        Physical dimensions of the rectangular fluid domain.
    occupancy_grid : list[list[float]]
        2D list representing the occupancy of the domain, where 0 indicates solid and 1 indicates fluid.Soft (0-1) CNN output before thresholding. Shape: (grid_ny, grid_nx).
        Stored so the CNN state can be reproduced exactly for the same geometry, and so the optimizer can use the same grid for mapping results back to the original geometry.
    obstacle_polygons : list[list[tuple[float, float]]]
        List of polygons representing the obstructions in the geometry. Each polygon is a list of (x, y) coordinates of its vertices.
    config_id : str
        UUID assigned at creation - ties a GeometryConfig to its .msh file and simulation results for reproducibility and traceability.
    """

    grid_nx: int
    grid_ny: int
    domain_length: float
    domain_height: float
    occupancy_grid: list # shape (grid_ny, grid_nx), values between 0 and 1
    obstacle_polygons: list # list of polygons, where each polygon is a list of (x, y) vertex coordinates
    config_id: str = field(default_factory=lambda: str(uuid.uuid4()))

    # Serialization
    def save(self, path: str | Path) -> None:
        """Save to a JSON file. Mesh files use the same config_id stem."""
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2)
    
    @classmethod
    def load(cls, path: str | Path) -> "GeometryConfig":
        with open(path) as f:
            data = json.load(f)
        return cls(**data)
    
    @property
    def mesh_filename(self) -> str:
        """Canonical filename derived from config_id."""
        return f"hx_{self.config_id}.msh"
    
    def grid_cell_size(self) -> tuple[float, float]:
        """Physical size of each grid cell (dx, dy)."""
        return (self.domain_length / self.grid_nx, self.domain_height / self.grid_ny)
    
    def cell_center(self, ix: int, iy: int) -> tuple[float, float]:
        """
        Physical (x, y) coordinates of the center of grid cell (ix, iy).
        ix in [0, grid_nx), iy in [0, grid_ny).
        Origin is at the bottome left of the domain
        """
        dx, dy = self.grid_cell_size()
        x = (ix + 0.5) * dx
        y = (iy + 0.5) * dy - self.domain_height / 2
        return (x, y)

In [14]:
class HeatExchangerCNN(nn.Module):
    """
    Maps a latent vector z -> occupancy grid of shape (1, grid_ny, grid_nx).

    The output is passed through sigmoid so values are in (0, 1).
    During training, keep the soft output for RL policy log-probs.
    For geometry generation, threshold at 'threshold' to get binary cells,
    then extract connected blobs as obstacle polygons.

    Architecture: fully-connected projection -> reshape -> series of 
    transposed conv upsampling blocks. The final spatial size is determined by grid_nx and grid_ny
    - the network is built to match these exactly.    
    """

    def __init__(
            self,
            latent_dim: int = 32,
            grid_nx: int = 20,
            grid_ny: int = 10,
            base_channels: int = 64,
    ):
        super().__init__()
        self.latent_dim = latent_dim
        self.grid_nx = grid_nx
        self.grid_ny = grid_ny
        self.base_channels = base_channels

        # Project z to a small spatial feature map, then upsample.
        # We start at (base_channels, 2, 2) and double the spatial dims each block.
        # Number of upsampling blocks needed:
        self._n_ups_x = int(np.ceil(np.log2(grid_nx))) - 1 # number of doublings needed
        self._n_ups_y = int(np.ceil(np.log2(grid_ny))) - 1
        n_blocks = max(self._n_ups_x, self._n_ups_y)

        # FC projection
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, base_channels * 4),
            nn.ReLU(),
            nn.Linear(base_channels * 4, base_channels * 2 * 2),
            nn.ReLU()
        )
        self.init_channels = base_channels
        self.init_h = 2,
        self.init_w = 2

        # Upsampling blocks
        blocks = []
        in_ch = base_channels
        for i in range(n_blocks):
            out_ch = max(in_ch // 2, 8)
            blocks.append(nn.Sequential(
                nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(),
            ))
            in_ch = out_ch
        self.ups = nn.ModuleList(blocks)

        # Final 1x1 conv to singe-channel occupancy grid
        self.head = nn.Conv2d(in_ch, 1, kernel_size=1)
    
    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """
        Parameters:
        z: Tensor of shape (B, latent_dim)
        
        Returns: 
        Tensor of shape (B, 1, grid_ny, grid_nx) occupancy grid with values in (0, 1)
        """
        B = z.shape[0]
        x = self.fc(z) # (B, base_channels * 2 * 2)
        x = x.view(B, self.init_channels, self.init_h, self.init_w) # (B, base_channels, 2, 2)

        for up in self.ups:
            x = up(x) # upsample
        x = self.head(x) # (B, 1, H, W)
        
        # Crop or interpolate to extract target size
        x = torch.nn.functional.interpolate(
            x, size=(self.grid_ny, self.grid_nx), mode='bilinear', align_corners=False
            )
        x = torch.sigmoid(x) # values in (0, 1)
        return x

In [15]:
class HeatExchanger:
    """
    Contains the geometry, mesh, and simulation results for a single heat exchanger design. This is the main data structure that the Generator produces and the Optimizer consumes.

    Parameters:
    id: str
        Unique identifier for this heat exchanger design, typically derived from the GeometryConfig's config_id.
    config_filename: Optional[str]
        Path to the GeometryConfig JSON file that describes the geometry of this heat exchanger. This file is used to generate the mesh and to map simulation results back to the original grid.
    mesh_filename: Optional[str]
        Path to the GMSH .msh file that contains the mesh for this heat exchanger design. This file is generated from the GeometryConfig and is used as input for the simulation.
    result_filename: Optional[str]
        Path to the simulation results file (e.g., Exodus .e file) for this heat exchanger design. This file is generated by running the simulation on the mesh and contains the results that will be analyzed and used for optimization.
    geometry_config: Optional[GeometryConfig]
        An instance of GeometryConfig that describes the geometry of this heat exchanger. This is typically loaded from the config_filename and is used for reference throughout the optimization process.
    obstruction_polygons: Optional[list]
        A list of polygons representing the obstructions in the geometry, extracted from the occupancy grid. Each polygon is a list of (x, y) coordinates of its vertices. This is derived from the GeometryConfig and is used for mesh generation and visualization.
    solved: bool
        A flag indicating whether the simulation for this heat exchanger design has been run and results are available. This is used to track the state of the optimization process.
    results: Optional[dict]
        A dictionary containing the analyzed results from the simulation, such as temperature distribution, flow patterns, and performance metrics. This is derived from the result_filename and is used for optimization and decision-making.
    """
    
    def __init__(
        self,
        id: str,
        mesh_params: dict,
        geometry_config: Optional[GeometryConfig] = None,
    ):
        self.id = id
        self.config_filename = None
        self.mesh_filename = None
        self.result_filename = None
        self.set_filenames()
        
        # Load geometry config and extract obstruction polygons
        self.geometry_config = geometry_config
        self.obstruction_polygons = None
        if geometry_config is None:
            self.load_geometry_config()
        else:
            self.obstruction_polygons = self.geometry_config.obstacle_polygons

        # Check for existing mesh and results
        self.mesh_exists = self.check_for_mesh()
        
        # Simulation state
        self.results = None
        self.solved = self.check_for_existing_results()
        self.mesh_params = mesh_params
    
    def set_filenames(self):
        """Set the mesh_filename and result_filename based on the heat exchanger ID."""
        self.config_filename = f"configs/hx_{self.id}.json"
        self.mesh_filename = f"meshes/hx_{self.id}.msh"
        self.result_filename = f"results/hx_{self.id}.e"

    def load_geometry_config(self):
        """Load the GeometryConfig from the config_filename."""
        self.geometry_config = GeometryConfig.load(self.config_filename)
        self.obstruction_polygons = self.geometry_config.obstacle_polygons

    def check_for_mesh(self) -> bool:
        """Check if the mesh file already exists for this heat exchanger design."""
        if os.path.exists(self.mesh_filename):
            return True
        else:
            self.generate_mesh()
        return os.path.exists(self.mesh_filename)

    def check_for_existing_results(self) -> bool:
        """Check if the result file already exists for this heat exchanger design."""
        if os.path.exists(self.result_filename):
            self.read_results()
        return os.path.exists(self.result_filename)
    
    def read_results(self):
        """Read the simulation results from the result_filename and store them in self.results."""
        # Placeholder for reading results - this would depend on the format of the results file
        self.results = None

    def create_custom_obstruction(self, points: list[tuple[float, float]]) -> int:
        """
        Create a planar surface in GMSH from an ordered list of (x, y) points.
        Returns the GMSH surface tag.
        """
        n_points = len(points)
        gmsh_points = []
        for coord in points:
            pt = gmsh.model.occ.addPoint(coord[0], coord[1], 0)
            gmsh_points.append(pt)
 
        lines = []
        for i in range(n_points):
            line = gmsh.model.occ.addLine(
                gmsh_points[i],
                gmsh_points[(i + 1) % n_points],
            )
            lines.append(line)
 
        cl = gmsh.model.occ.addCurveLoop(lines)
        surface = gmsh.model.occ.addPlaneSurface([cl])
        return surface

    def generate_mesh(self) -> str:
        """
        Returns the path to the written .msh file.
        """
        gmsh.initialize()
        gmsh.model.add(f"hx_{self.config.config_id}")
 
        rect = gmsh.model.occ.addRectangle(
            0, 0, 0,
            self.domain_length, self.domain_height,
        )
        gmsh.model.occ.synchronize()
 
        obstructions = []
        for polygon in self.obstruction_polygons:
            surface = self.create_custom_obstruction(polygon)
            obstructions.append(surface)
        gmsh.model.occ.synchronize()
 
        if obstructions:
            cut = gmsh.model.occ.cut(
                [(2, rect)],
                [(2, s) for s in obstructions],
                removeObject=True,
                removeTool=True,
            )
            gmsh.model.occ.synchronize()
            out_dim_tags, _ = cut
            fluid_surfaces = [tag for dim, tag in out_dim_tags if dim == 2]
        else:
            # No obstacles — entire rectangle is the fluid domain
            fluid_surfaces = [rect]
 
        if not fluid_surfaces:
            gmsh.finalize()
            raise RuntimeError(
                "Boolean cut produced no fluid domain. "
                "Check that obstacles don't fill the entire domain."
            )
 
        gmsh.model.addPhysicalGroup(2, fluid_surfaces, name="Fluid")

        # Boundary classification
        gmsh.model.occ.synchronize()
        boundaries = gmsh.model.getBoundary(
            [(2, s) for s in fluid_surfaces], oriented=False
        )
 
        wall_curves, inlet, outlet, top, bottom = [], [], [], [], []
        for dim, tag in boundaries:
            x, y, _ = gmsh.model.occ.getCenterOfMass(dim, tag)
            if abs(x) < 1e-6:
                inlet.append(tag)
            elif abs(x - self.domain_length) < 1e-6:
                outlet.append(tag)
            elif abs(y - self.domain_height / 2) < 1e-6:
                top.append(tag)
            elif abs(y + self.domain_height / 2) < 1e-6:
                bottom.append(tag)
            else:
                wall_curves.append(tag)
 
        if wall_curves:
            gmsh.model.addPhysicalGroup(1, wall_curves, name="Wall")
        if inlet:
            gmsh.model.addPhysicalGroup(1, inlet, name="Inlet")
        if outlet:
            gmsh.model.addPhysicalGroup(1, outlet, name="Outlet")
        if top:
            gmsh.model.addPhysicalGroup(1, top, name="Top")
        if bottom:
            gmsh.model.addPhysicalGroup(1, bottom, name="Bottom")
 
        p = self.mesh_params
        gmsh.option.setNumber("Mesh.CharacteristicLengthMin", p["mesh_min"])
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", p["mesh_max"])
        gmsh.option.setNumber("Mesh.Algorithm",               p["mesh_algorithm"])
        gmsh.option.setNumber("Mesh.RecombineAll",             p["mesh_recombine"])
        gmsh.option.setNumber("Mesh.ElementOrder",             p["mesh_element_order"])
 
        gmsh.model.mesh.generate(2)
        gmsh.write(self.mesh_filename)
        gmsh.finalize()

        return self.mesh_filename

In [16]:
class HeatExchangerGenerator:
    """
    Wraps the CNN and all geometry post-processing.

    Used by the Optimizer to:
    - Sample a latent vector z (or receive one from the RL policy).
    - Produce a GeometryConfig (occupancy grid + polygon list + metadata)
    - Pass the config to MeshGenerator to write a .msh file

    Parameters:
    grid_nx, grid_ny: int
        Resolution of the occupancy grid used for geometry representation and CNN output.
    domain_length, domain_height: float
        Physical dimensions of the heat exchanger domain (in meters or consistent units).
    latent_dim: int
        Dimensions of the CNN latent space.
    threshold: float
        Threshold for converting the CNN's soft occupancy output into binary solid/fluid classification.
    device: str
        PyTorch device to run the CNN on ('cuda', 'mps', or 'cpu').
    mesh_params: dict | None
        Optional GMSH meshing parameters to forward to MeshGenerator. If None, defaults will be used.
    """

    def __init__(
        self,
        grid_nx: int = 20,
        grid_ny: int = 10,
        domain_length: float = 1.0,
        domain_height: float = 0.5,
        latent_dim: int = 32,
        threshold: float = 0.5,
        device: str = "cpu",
        mesh_params: Optional[dict] = None,
    ):
        self.grid_nx = grid_nx
        self.grid_ny = grid_ny
        self.domain_length = domain_length
        self.domain_height = domain_height
        self.latent_dim = latent_dim
        self.threshold = threshold
        self.device = torch.device(device)

        self.mesh_params = mesh_params or {
            "mesh_min": 0.01,
            "mesh_max": 0.05,
            "mesh_algorithm": 8,
            "mesh_recombine": 1,
            "mesh_element_order": 2
        }

        self.cnn = HeatExchangerCNN(
            latent_dim=latent_dim,
            grid_nx=grid_nx,
            grid_ny=grid_ny,
            base_channels=64
        ).to(self.device)

        self.heat_exchangers = []

    def occupancy_to_polygons(
            self,
            occupancy: np.ndarray, # shape (grid_ny, grid_nx), values in [0, 1]
            domain_length: float,
            domain_height: float,
            threshold: float = 0.5,
            min_cells: int = 1,
    ) -> list[list[tuple[float, float]]]:
        """
        Convert a thresholded occupancy grid into a list of rectangular obstacle polygons, one per connected solid region.
        
        Each obstacle is the bounding box of a connencted componenent - this keeps the polygons valid for GMSH
        and avoids staircase artifacts that would cause meshing failures.

        Parameters:
        occupancy: np.ndarray
            Soft occupancy grid, shape (grid_ny, grid_nx), values in [0, 1]
        domain_length, domain_height: float
            Length and height of the domain
        threshold: float = 0.5
            Threshold for binarizing the occupancy grid
        min_cells: int = 1
            Minimum number of cells for a connected component to be considered an obstacle

        Returns:
        list[list[tuple[float, float]]]
            List of obstacle polygons, each represented as a list of (x, y) coordinates
        """
        grid_ny, grid_nx = occupancy.shape
        dx = domain_length / grid_nx
        dy = domain_height / grid_ny

        binary = (occupancy >= threshold).astype(int)
        labeled, n_components = label(binary)

        polygons = []
        for comp_id in range(1, n_components + 1):
            cells = np.argwhere(labeled == comp_id)
            if len(cells) < min_cells:
                continue

            # Bounding box in grid indices
            iy_min, ix_min = cells.min(axis=0)
            iy_max, ix_max = cells.max(axis=0)

            # Convert to physical coordinates (y is measured from the bottom)
            x0 = ix_min * dx
            x1 = (ix_max + 1) * dx
            y0 = iy_min * dy
            y1 = (iy_max + 1) * dy

            # CCW rectangle
            polygon = [(x0, y0), (x1, y0), (x1, y1), (x0, y1)]
            polygons.append(polygon)
        
        return polygons

    def generate(self, 
                 z: Optional[torch.Tensor] = None, 
                 config_id: Optional[str] = None
                 ) -> tuple["GeometryConfig", torch.Tensor]:
        """
        Generate a heat exchanger geometry from a latent vector z.

        Parameters:
        z: Optional[torch.Tensor]
            Latent vector of shape (1, latent_dim). If None, a random vector will be sampled from a standard normal distribution.
        config_id: Optional[str]
            UUID string to assign to the GeometryConfig. If None, a new UUID will be generated.

        Returns:
        GeometryConfig
            The generated geometry configuration containing the occupancy grid, obstacle polygons, and metadata.
        torch.Tensor
            The raw soft occupancy grid output from the CNN before thresholding, shape (1, grid_ny, grid_nx).
        """
        if z is None:
            z = torch.randn(1, self.latent_dim, device=self.device)
        
        self.cnn.eval()
        with torch.no_grad():
            soft_grid = self.cnn(z) # (1, 1, grid_ny, grid_nx)

        occupancy_np = soft_grid.squeeze().cpu().numpy() # (grid_ny, grid_nx)
        
        obstacle_polygons = self.occupancy_to_polygons(
            occupancy=occupancy_np,
            domain_length=self.domain_length,
            domain_height=self.domain_height,
            threshold=self.threshold,
            min_cells=1
        )

        config = GeometryConfig(
            grid_nx=self.grid_nx,
            grid_ny=self.grid_ny,
            domain_length=self.domain_length,
            domain_height=self.domain_height,
            occupancy_grid=occupancy_np.tolist(),
            obstacle_polygons=obstacle_polygons,
            config_id=config_id or str(uuid.uuid4()),
        )

        hx = HeatExchanger(
            id=config.config_id,
            mesh_params=self.mesh_params,
            geometry_config=config
        )
        self.heat_exchangers.append(hx)

        return config, soft_grid
    
    def generate_batch(self, n: int) -> list["GeometryConfig"]:
        """
        Generate a batch of heat exchanger geometries.

        Parameters:
        n: int
            Number of geometries to generate.

        Returns:
        list[GeometryConfig]
            List of generated geometry configurations.
        """
        configs = []
        for _ in range(n):
            config, _ = self.generate()
            configs.append(config)
        return configs
    
    # Additional methods for training the CNN, saving/loading model weights, etc.
    def save_weights(self, path: str | Path) -> None:
        torch.save(self.cnn.state_dict(), path)

    def load_weights(self, path: str | Path) -> None:
        self.cnn.load_state_dict(torch.load(path, map_location=self.device))
    
    def parameters(self):
        # Expose CNN parameters for optimization/training
        return self.cnn.parameters()

In [17]:
hx_gen = HeatExchangerGenerator(
    grid_nx=20,
    grid_ny=10,
    domain_length=1.0,
    domain_height=0.5,
    latent_dim=32,
    threshold=0.5,
    device="cuda" if torch.cuda.is_available() else "cpu",
    mesh_params={
        "mesh_min": 0.01,
        "mesh_max": 0.05,
        "mesh_algorithm": 8,
        "mesh_recombine": 1,
        "mesh_element_order": 2
    }
)

In [18]:
hx_gen.generate_batch(5)

TypeError: view(): argument 'size' failed to unpack the object at pos 3 with error "type must be tuple of ints,but got tuple"